# Weyl Tensor & Kretschner Scalar — Curvature Invariants

This tutorial computes two important curvature invariants of spacetime:

- **Weyl tensor** $C_{abcd}$: the trace-free part of the Riemann tensor, encoding tidal (conformal) curvature.
- **Kretschner scalar** $K = R_{abcd}R^{abcd}$: a coordinate-invariant measure of curvature magnitude that can distinguish true singularities from coordinate artifacts.

In [1]:
import os
from symbolica import set_license_key

_key = os.environ.get("SYMBOLICA_LICENSE", "")
if _key:
    set_license_key(_key)

from gravica import (
    WeylTensor,
    RiemannTensor,
    kretschner_scalar,
    display,
)
from gravica.simplify import simplify, str_is_zero
from gravica.metrics import schwarzschild

## 1. Weyl Tensor of the Schwarzschild Spacetime

The Schwarzschild solution is a **vacuum solution** ($R_{ab} = 0$), so the entire Riemann tensor reduces to the Weyl tensor:

$$
C_{abcd} = R_{abcd} - \frac{2}{n-2}\bigl(g_{a[c}R_{d]b} - g_{b[c}R_{d]a}\bigr) + \frac{2}{(n-1)(n-2)}R\,g_{a[c}g_{d]b}
$$

In vacuum ($R_{ab} = 0$, $R = 0$) this simplifies to $C_{abcd} = R_{abcd}$.

In [2]:
g = schwarzschild()
coords = ["t", "r", "\\theta", "\\varphi"]

weyl = WeylTensor.from_metric(g)

print("Non-zero independent components:")
display.components_table(display.nonzero_components(weyl, coords), weyl)

Non-zero independent components:
┌────────────────────────────────────────────────────────┐
│ You are running a restricted Symbolica instance.       │
│                                                        │
│ This mode is only permitted for non-commercial use and │
│ is limited to one instance and core per machine.       │
│                                                        │
│ Hobbyists can easily acquire a free license key        │
│ that unlocks all cores and removes this banner:        │
│                                                        │
│   from symbolica import *                              │
│   request_hobbyist_license('YOUR_NAME', 'YOUR_EMAIL')  │
│                                                        │
│ All other users can obtain a free 30-day trial key:    │
│                                                        │
│   from symbolica import *                              │
│   request_trial_license('NAME', 'EMAIL', 'EMPLOYER')   │
│                      

| Component | Value |
|-----------|-------|
| $C^{t}{}_{r t r}$ | $\frac{r_s}{r^{3}}$ |
| $C^{t}{}_{\theta t \theta}$ | $\frac{\frac{1}{2} \left(-r r_s+r_s^{2}\right)}{r^{2}}$ |
| $C^{t}{}_{\varphi t \varphi}$ | $\frac{\frac{1}{2} \left(-r r_s \sin\!\left(\theta\right)^{2}+r_s^{2} \sin\!\left(\theta\right)^{2}\right)}{r^{2}}$ |
| $C^{r}{}_{t t r}$ | $\frac{-r_s}{r^{3}}$ |
| $C^{r}{}_{\theta r \theta}$ | $\frac{-r_s}{-2 r+2 r_s}$ |
| $C^{r}{}_{\varphi r \varphi}$ | $\frac{-r_s \sin\!\left(\theta\right)^{2}}{-2 r+2 r_s}$ |
| $C^{\theta}{}_{t t \theta}$ | $\frac{\frac{1}{2} \left(r r_s-r_s^{2}\right)}{r^{2}}$ |
| $C^{\theta}{}_{r r \theta}$ | $\frac{r_s}{-2 r+2 r_s}$ |
| $C^{\theta}{}_{\varphi \theta \varphi}$ | $-r r_s \sin\!\left(\theta\right)^{2}$ |
| $C^{\varphi}{}_{t t \varphi}$ | $\frac{\frac{1}{2} \left(r r_s \sin\!\left(\theta\right)^{2}-r_s^{2} \sin\!\left(\theta\right)^{2}\right)}{r^{2}}$ |
| $C^{\varphi}{}_{r r \varphi}$ | $\frac{r_s \sin\!\left(\theta\right)^{2}}{-2 r+2 r_s}$ |
| $C^{\varphi}{}_{\theta \theta \varphi}$ | $r r_s \sin\!\left(\theta\right)^{2}$ |

## 2. Kretschner Scalar — Detecting Singularities

The Kretschner scalar is a coordinate-invariant quantity, so it can distinguish **coordinate singularities** (removable by a change of coordinates) from **true singularities** (where curvature genuinely diverges).

For the Schwarzschild spacetime:
- $r = r_s$: **coordinate singularity** ($K$ is finite — not a real singularity)
- $r = 0$: **true singularity** ($K \to \infty$)

The expected result is $K = 12\,r_s^2 / r^6$.

In [3]:
riemann = RiemannTensor.from_metric(g)
K = kretschner_scalar(riemann)

print("Kretschner scalar K = R_{abcd} R^{abcd}:")
print(K)

Kretschner scalar K = R_{abcd} R^{abcd}:
12*r_s^2/r^6


The result $K = 12\,r_s^2 / r^6$ diverges as $r \to 0$ but remains finite at $r = r_s$,
confirming that only $r = 0$ is a true singularity.

## 3. Verifying Weyl = Riemann in Vacuum

We verify that $C_{abcd} = R_{abcd}$ holds for this vacuum solution.

In [4]:
n = g.dim
all_match = True
for a in range(n):
    for b in range(n):
        for c in range(n):
            for d in range(c + 1, n):
                diff = simplify(weyl[a, b, c, d] - riemann.fully_covariant(a, b, c, d))
                if not str_is_zero(diff):
                    all_match = False
                    print(f"C_{a}{b}{c}{d} != R_{a}{b}{c}{d}: diff = {diff}")

if all_match:
    print("Verified: C_{abcd} = R_{abcd} in vacuum.")

Verified: C_{abcd} = R_{abcd} in vacuum.
